In [1]:
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.common.exceptions import TimeoutException
from datetime import datetime, timedelta



# SETUP CHROME


options = Options()

# Untuk melihat proses scraping
# Kalau sudah yakin berjalan, bisa aktifkan:
# options.add_argument('--headless')

options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')
options.add_argument('--window-size=1920,1080')

# Tidak menunggu seluruh resource selesai loading
options.page_load_strategy = 'eager'


driver = webdriver.Chrome(
    service=Service(),
    options=options
)

# Timeout maksimal untuk driver.get()
driver.set_page_load_timeout(20)



# MEMBERSIHKAN DESKRIPSI


def bersihkan_deskripsi_dan_lokasi(deskripsi):

    if not deskripsi:
        return None, None

    # Normalisasi line break
    teks = deskripsi.replace('\r\n', '\n').replace('\r', '\n')

    # Cari bagian "Permasalahan:"
    if 'Permasalahan:' in teks:

        teks = teks.split(
            'Permasalahan:',
            1
        )[1]

    # Pisahkan bagian Kategori
    if 'Kategori:' in teks:

        bagian = teks.split(
            'Kategori:',
            1
        )

        deskripsi_bersih = bagian[0].strip()

        sisa = bagian[1]

    else:

        deskripsi_bersih = teks.strip()

        sisa = ''


    # AMBIL LOKASI DARI DESCRIPTION

    lokasi = None

    if 'Lokasi:' in sisa:

        lokasi = sisa.split(
            'Lokasi:',
            1
        )[1].strip()

    # Bersihkan jika masih ada whitespace berlebih
    deskripsi_bersih = ' '.join(
        deskripsi_bersih.split()
    )

    if lokasi:

        lokasi = ' '.join(
            lokasi.split()
        )


    return deskripsi_bersih, lokasi



# SCRAPE SATU LAPORAN


def scrape_laporan(driver, nomor_laporan):

    url = f'https://crm.jakarta.go.id/laporan/{nomor_laporan}'

    try:

        print(f"\nMembuka: {nomor_laporan}")

        driver.get(url)

        # Tunggu sampai description ATAU status
        # memiliki isi
        try:

            WebDriverWait(driver, 10).until(
                lambda d: (
                    d.find_element(
                        By.ID,
                        'description'
                    ).text.strip() != ''
                    or
                    d.find_element(
                        By.ID,
                        'status'
                    ).text.strip() != ''
                )
            )

        except TimeoutException:

            print(
                f"⚠ {nomor_laporan}: "
                f"data belum muncul setelah 10 detik"
            )

            return None


        # AMBIL DATA DARI WEBSITE

        data = {}

        data['nomor_laporan'] = nomor_laporan


        # Description mentah
        try:

            deskripsi_raw = driver.find_element(
                By.ID,
                'description'
            ).text.strip()

        except:

            deskripsi_raw = None


        # Kategori
        try:

            kategori = driver.find_element(
                By.ID,
                'category'
            ).text.strip()

        except:

            kategori = None


        # Tanggal
        try:

            tanggal = driver.find_element(
                By.ID,
                'dateOfEntry'
            ).text.strip()

        except:

            tanggal = None

        # Lokasi
        try:

            lokasi_website = driver.find_element(
                By.ID,
                'location'
            ).text.strip()

        except:

            lokasi_website = None


        # Status
        try:

            status_laporan = driver.find_element(
                By.ID,
                'status'
            ).text.strip()

        except:

            status_laporan = None


        # BERSIHKAN DESCRIPTION

        deskripsi_laporan, lokasi_dari_deskripsi = (
            bersihkan_deskripsi_dan_lokasi(
                deskripsi_raw
            )
        )


        # TENTUKAN LOKASI

        # Prioritas:
        # lokasi yang ada di dalam description
        # karena itu merupakan lokasi kejadian.

        if lokasi_dari_deskripsi:

            lokasi = lokasi_dari_deskripsi

        else:

            # Kalau format description berbeda dan
            # tidak memiliki "Lokasi:", gunakan field
            # website sebagai fallback.
            lokasi = lokasi_website


        # BUAT DATA

        data['deskripsi_laporan'] = deskripsi_laporan
        data['kategori'] = kategori
        data['tanggal'] = tanggal
        data['lokasi'] = lokasi
        data['status_laporan'] = status_laporan


        # CEK DATA

        if not any([
            data['deskripsi_laporan'],
            data['kategori'],
            data['tanggal'],
            data['lokasi'],
            data['status_laporan']
        ]):

            print(
                f"✗ {nomor_laporan}: "
                f"tidak ada data"
            )

            return None


        # BERHASIL

        print(
            f"✓ {nomor_laporan} berhasil"
        )

        print(
            f"  Kategori : {data['kategori']}"
        )

        print(
            f"  Tanggal  : {data['tanggal']}"
        )

        print(
            f"  Lokasi   : {data['lokasi']}"
        )

        print(
            f"  Status   : {data['status_laporan']}"
        )


        return data



    # PAGE LOAD TIMEOUT


    except TimeoutException:

        print(
            f"⚠ {nomor_laporan}: "
            f"page load timeout"
        )

        try:

            print(
                "→ Mencoba membaca data yang "
                "sudah tersedia..."
            )

            data = {}

            data['nomor_laporan'] = nomor_laporan


            try:

                deskripsi_raw = driver.find_element(
                    By.ID,
                    'description'
                ).text.strip()

            except:

                deskripsi_raw = None


            try:

                data['kategori'] = driver.find_element(
                    By.ID,
                    'category'
                ).text.strip()

            except:

                data['kategori'] = None


            try:

                data['tanggal'] = driver.find_element(
                    By.ID,
                    'dateOfEntry'
                ).text.strip()

            except:

                data['tanggal'] = None


            try:

                lokasi_website = driver.find_element(
                    By.ID,
                    'location'
                ).text.strip()

            except:

                lokasi_website = None


            try:

                data['status_laporan'] = driver.find_element(
                    By.ID,
                    'status'
                ).text.strip()

            except:

                data['status_laporan'] = None


            # Bersihkan description
            (
                data['deskripsi_laporan'],
                lokasi_dari_deskripsi
            ) = bersihkan_deskripsi_dan_lokasi(
                deskripsi_raw
            )


            if lokasi_dari_deskripsi:

                data['lokasi'] = lokasi_dari_deskripsi

            else:

                data['lokasi'] = lokasi_website


            if any([
                data['deskripsi_laporan'],
                data['kategori'],
                data['tanggal'],
                data['lokasi'],
                data['status_laporan']
            ]):

                print(
                    f"✓ {nomor_laporan}: "
                    f"data berhasil dibaca setelah timeout"
                )

                return data

        except:

            pass


        return None



# SETUP SCRAPING


results = []

# TARGET JUMLAH LAPORAN
target_laporan = 100

# Mulai dari tanggal
current_date = datetime(2026, 9, 1)

# Index laporan dalam satu hari
index_per_day = 0



# SCRAPING OTOMATIS


try:

    while len(results) < target_laporan:

        day_prefix = current_date.strftime('%y%m%d')

        failure_count = 0


        while (
            failure_count < 10
            and len(results) < target_laporan
        ):

            nomor_laporan = (
                f'JK{day_prefix}'
                f'{str(index_per_day).zfill(4)}'
            )


            laporan = scrape_laporan(
                driver,
                nomor_laporan
            )


            index_per_day += 1



            # BERHASIL


            if laporan:

                results.append(laporan)

                failure_count = 0

                print(
                    f"→ Total berhasil: "
                    f"{len(results)}/{target_laporan}"
                )



            # GAGAL


            else:

                failure_count += 1

                print(
                    f"✗ Gagal/tidak ditemukan "
                    f"({failure_count}/10)"
                )


    
        # PINDAH HARI
    

        if failure_count >= 10:

            print(
                f"\n10 laporan berturut-turut "
                f"tidak ditemukan."
            )

            print(
                f"Pindah dari "
                f"{current_date.strftime('%d-%m-%Y')} "
                f"ke tanggal berikutnya.\n"
            )

            current_date += timedelta(days=1)

            index_per_day = 0


finally:

    driver.quit()



# DATAFRAME


df = pd.DataFrame(results)


df = df[
    [
        'nomor_laporan',
        'deskripsi_laporan',
        'kategori',
        'tanggal',
        'lokasi',
        'status_laporan'
    ]
]



# HASIL


print("\n")
print("-" * 5)
print("HASIL SCRAPING")
print("-" * 5)

print(df)

print("\nJumlah laporan:", len(df))



# SIMPAN CSV


df.to_csv(
    'data_laporan_crm.csv',
    index=False,
    encoding='utf-8-sig'
)

print(
    "\n✓ Data berhasil disimpan sebagai "
    "'data_laporan_crm.csv'"
)


Membuka: JK2609010000
⚠ JK2609010000: data belum muncul setelah 10 detik
✗ Gagal/tidak ditemukan (1/10)

Membuka: JK2609010001
✓ JK2609010001 berhasil
  Kategori : Jaringan Utilitas
  Tanggal  : 01 Sep 2026
  Lokasi   : Tambora, Jalan Jembatan Dua Flyover, West Jakarta, Jakarta, ID
  Status   : Selesai
→ Total berhasil: 1/100

Membuka: JK2609010002
⚠ JK2609010002: data belum muncul setelah 10 detik
✗ Gagal/tidak ditemukan (1/10)

Membuka: JK2609010003
⚠ JK2609010003: data belum muncul setelah 10 detik
✗ Gagal/tidak ditemukan (2/10)

Membuka: JK2609010004
⚠ JK2609010004: data belum muncul setelah 10 detik
✗ Gagal/tidak ditemukan (3/10)

Membuka: JK2609010005
⚠ JK2609010005: data belum muncul setelah 10 detik
✗ Gagal/tidak ditemukan (4/10)

Membuka: JK2609010006
⚠ JK2609010006: data belum muncul setelah 10 detik
✗ Gagal/tidak ditemukan (5/10)

Membuka: JK2609010007
⚠ JK2609010007: data belum muncul setelah 10 detik
✗ Gagal/tidak ditemukan (6/10)

Membuka: JK2609010008
✓ JK2609010008 ber